# Federated Training — FedAvg / FedBN on MAMA-MIA

**Purpose:** Phase 2 federated learning pipeline. Architecture-agnostic — the same notebook trains UNet3D baseline, FADC-Bottleneck, FADC-Deep, or any future variant by switching `MODEL` in the CONFIG cell.

**Two algorithms supported in v1:**
- `fedavg` — weighted average of all parameters across clients.
- `fedbn` — same as FedAvg but BatchNorm state is kept per-client (strong fit for multi-site MRI).

**Two partitions:**
- `natural` — 4 clients = DUKE / ISPY1 / ISPY2 / NACT (paper headline)
- `iid` — random shuffle into K equal shards (FL infrastructure sanity check)

**Centralized references (locked):**
- 3D U-Net 2ch baseline: 0.6735 training-val Dice (0.7036 post-TTA)
- FADC-Deep 2ch: 0.6863 training-val Dice (0.7131 post-TTA — new paper headline)

**IMPORTANT:** Download `best_model.pth`, `fl_log.json`, `meta.json` from `OUTPUT_DIR` before `/kaggle/working` wipes.

In [ ]:
# ────────────────────────────────
# CONFIGURATION — edit these before each run
# ────────────────────────────────
MODEL           = "unet3d"         # unet3d | unet3d_fadc_bottleneck | unet3d_fadc_deep | ...
ALGORITHM       = "fedavg"         # fedavg | fedbn
PARTITION       = "natural"        # natural | iid
N_CLIENTS       = 4                # ignored for natural; sets shard count for iid

ROUNDS          = 5                # quick smoke to see the pipeline working end-to-end
LOCAL_EPOCHS    = 2                # E in standard FL notation
CLIENT_FRACTION = 1.0              # 1.0 = all clients each round; 0.5 = sample half
LR              = 1e-4
SEED            = 42

BATCH_SIZE      = 2
NUM_WORKERS     = 4
PATCH_SIZE      = [128, 128, 64]

PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
DATA_ROOT     = PREPROCESSED_CACHE_DIR
OUTPUT_DIR    = f"/kaggle/working/outputs/fl_{MODEL}_{ALGORITHM}_{PARTITION}_R{ROUNDS}_E{LOCAL_EPOCHS}_s{SEED}"
CODE_DIR      = "/kaggle/working/FADC-3D"

print(f"MODEL        : {MODEL}")
print(f"ALGORITHM    : {ALGORITHM}")
print(f"PARTITION    : {PARTITION} ({N_CLIENTS} clients for iid)")
print(f"ROUNDS       : {ROUNDS} | LOCAL_EPOCHS: {LOCAL_EPOCHS} | CLIENT_FRACTION: {CLIENT_FRACTION}")
print(f"SEED         : {SEED}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")

In [ ]:
# ────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("Dependencies ready.")

In [ ]:
# ────────────────────────────────
# 2. CLONE / UPDATE CODE  (feature/fl-baseline branch — isolated from any active FADC v2.2 training run)
# ────────────────────────────────
import os, sys
FL_BRANCH = "feature/fl-baseline"
if os.path.exists(CODE_DIR):
    print(f"Repo exists — pulling latest from {FL_BRANCH}")
    os.system(f"git -C {CODE_DIR} fetch origin {FL_BRANCH}")
    os.system(f"git -C {CODE_DIR} checkout {FL_BRANCH}")
    os.system(f"git -C {CODE_DIR} pull origin {FL_BRANCH}")
else:
    os.system(f"git clone -b {FL_BRANCH} https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
sys.path.insert(0, CODE_DIR)
print(f"Code path: {CODE_DIR}")
os.system(f"git -C {CODE_DIR} log --oneline -1")

In [ ]:
# ────────────────────────────────
# 3. SANITY CHECK — cache + partition preview
# ────────────────────────────────
import os, numpy as np
from pathlib import Path

cache_path = Path(PREPROCESSED_CACHE_DIR)
assert cache_path.exists(), f"Cache path not found: {cache_path}"
train_npzs = sorted((cache_path / "train").glob("*.npz"))
val_npzs   = sorted((cache_path / "val").glob("*.npz"))
print(f"Train .npz files : {len(train_npzs)}")
print(f"Val   .npz files : {len(val_npzs)}")
assert len(train_npzs) > 0 and len(val_npzs) > 0

d = np.load(train_npzs[0])
print(f"Sample shape  : image={d['image'].shape}  label={d['label'].shape}")
assert d["image"].shape[0] == 2, "FATAL: cache is not 2-channel"

from data.mama_mia_dataset import discover_cases_from_cache
from training.fl_partitions import partition_factory, partition_summary

split_csv = os.path.join(DATA_ROOT, "train_test_splits.csv")
if not os.path.exists(split_csv):
    split_csv = None
    print("(no split CSV — using all cached cases)")
train_cases = discover_cases_from_cache(str(cache_path / "train"), split_csv, split="train")
val_cases   = discover_cases_from_cache(str(cache_path / "val"),   split_csv, split="test")
print(f"Train cases discovered: {len(train_cases)} | Val: {len(val_cases)}")

partition_fn = partition_factory(PARTITION, N_CLIENTS, SEED)
partition    = partition_fn(train_cases)
print(f"\nPartition preview ({PARTITION}):")
print(partition_summary(partition))

In [ ]:
# ────────────────────────────────
# 4. LAUNCH FEDERATED TRAINING
# ────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

script = os.path.join(CODE_DIR, "scripts", "fl_simulate.py")
cmd = [
    sys.executable, "-u", script,
    "--model",           MODEL,
    "--algorithm",       ALGORITHM,
    "--partition",       PARTITION,
    "--n_clients",       str(N_CLIENTS),
    "--rounds",          str(ROUNDS),
    "--local_epochs",    str(LOCAL_EPOCHS),
    "--client_fraction", str(CLIENT_FRACTION),
    "--lr",              str(LR),
    "--batch_size",      str(BATCH_SIZE),
    "--num_workers",     str(NUM_WORKERS),
    "--patch_size",      str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--data_root",       DATA_ROOT,
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--seed",            str(SEED),
    "--output_dir",      OUTPUT_DIR,
]
print("Command:", " ".join(cmd))
print("=" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nExit code: {proc.returncode}")

In [ ]:
# ────────────────────────────────
# 5. PLOT — global Dice per round + per-client breakdown
# ────────────────────────────────
import json, os
import matplotlib.pyplot as plt

CENTRALIZED_BASELINE = 0.6735   # 3D U-Net 2ch locked baseline (uncontrolled seed)
DEEP_TTA_HEADLINE    = 0.7131   # FADC-Deep + TTA (paper headline)

log_path = os.path.join(OUTPUT_DIR, "fl_log.json")
if not os.path.exists(log_path):
    print("No fl_log.json found yet.")
else:
    with open(log_path) as f:
        log = json.load(f)

    rounds        = [r["round"]          for r in log]
    global_dice   = [r["global"]["dice"] for r in log]
    global_sens   = [r["global"]["sens"] for r in log]

    # Per-client Dice over rounds
    client_ids = sorted(log[0]["per_client"].keys(), key=lambda x: int(x))
    per_client_curves = {cid: [r["per_client"][cid]["dice"] for r in log] for cid in client_ids}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(rounds, global_dice, color="steelblue", linewidth=2, marker="o", markersize=4, label="Global Dice")
    ax1.axhline(y=CENTRALIZED_BASELINE, color="green", linestyle="--", linewidth=1.5,
                label=f"Centralized baseline ({CENTRALIZED_BASELINE:.4f})")
    ax1.axhline(y=DEEP_TTA_HEADLINE, color="red", linestyle="--", linewidth=1,
                label=f"FADC-Deep + TTA ({DEEP_TTA_HEADLINE:.4f})")
    ax1.set_title(f"FL Global Dice ({ALGORITHM} / {PARTITION} / seed={SEED})", fontsize=12)
    ax1.set_xlabel("Round")
    ax1.set_ylabel("Val Dice")
    ax1.legend(loc="lower right", fontsize=9)
    ax1.grid(True, alpha=0.3)

    for cid, curve in per_client_curves.items():
        ax2.plot(rounds, curve, linewidth=1.5, marker=".", markersize=3, label=f"Client {cid}")
    ax2.set_title("Per-Client Val Dice", fontsize=12)
    ax2.set_xlabel("Round")
    ax2.set_ylabel("Val Dice")
    ax2.legend(loc="lower right", fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fl_curves.png"), dpi=150)
    plt.show()

    best = max(global_dice)
    print(f"Rounds completed   : {len(log)}")
    print(f"Best global Dice   : {best:.4f}")
    print(f"Delta vs central   : {best - CENTRALIZED_BASELINE:+.4f} (centralized baseline 0.6735)")
    print(f"\nFinal per-client Dice:")
    for cid in client_ids:
        print(f"  Client {cid}: {per_client_curves[cid][-1]:.4f}")

In [ ]:
# ────────────────────────────────
# 6. CHECKPOINT DOWNLOAD HELPER
# Run after training finishes — zips outputs for one-click download from
# the Kaggle 'Output' tab (/kaggle/working files persist only until session close).
# ────────────────────────────────
import shutil, os
zip_base = OUTPUT_DIR.rstrip("/")
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Archive: {zip_path}")
for name in os.listdir(OUTPUT_DIR):
    p = os.path.join(OUTPUT_DIR, name)
    sz = os.path.getsize(p) / 1e6
    print(f"  {name:40s} {sz:8.2f} MB")